# GX_00_setup

Create schema and volume

In [0]:
   %sql
   CREATE SCHEMA IF NOT EXISTS workspace.churn;
   CREATE VOLUME IF NOT EXISTS workspace.churn.raw;

Version check and row counts

In [0]:
import sys

RAW = "/Volumes/workspace/churn/raw"

print("Spark:", spark.version)
print("Python:", sys.version.split()[0])

churn = (spark.read
        .option("header", True)
        .option("encoding", "ISO-8859-1")
        .csv(f"{RAW}/churn.csv"))

tx = (spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{RAW}/bank_transactions.csv"))

n_churn, n_tx = churn.count(), tx.count()
print("churn rows:", n_churn, "| cols:", len(churn.columns))
print("tx rows:   ", n_tx, "| cols:", len(tx.columns))

assert n_churn == 10_000, "churn row count wrong"
assert n_tx == 1_000, "transactions row count wrong"
print("Smoke test passed")

Install GraphFrames

In [0]:
%pip install graphframes-py

In [0]:
dbutils.library.restartPython()

Toy graph test

In [0]:
from graphframes import GraphFrame

v = spark.createDataFrame([("a",), ("b",), ("c",), ("d",), ("e",)], ["id"])
e = spark.createDataFrame([("a","b"), ("b","c"), ("c","a"), ("d","e")], ["src", "dst"])
g = GraphFrame(v, e)

print("inDegrees:")
g.inDegrees.show()

print("labelPropagation:")
g.labelPropagation(maxIter=5).show()

Test every GraphFrames function

In [0]:
tests = {
    "outDegrees":          lambda: g.outDegrees,
    "degrees":             lambda: g.degrees,
    "triplets":            lambda: g.triplets,
    "find (motif)":        lambda: g.find("(x)-[]->(y); (y)-[]->(z)"),
    "filterEdges":         lambda: g.filterEdges("src = 'a'"),
    "bfs":                 lambda: g.bfs("id = 'a'", "id = 'c'"),
    "connectedComponents": lambda: g.connectedComponents(),
    "pageRank":            lambda: g.pageRank(resetProbability=0.15, maxIter=5).vertices,
    "shortestPaths":       lambda: g.shortestPaths(landmarks=["a"]),
    "triangleCount":       lambda: g.triangleCount(),
    "stronglyConnected":   lambda: g.stronglyConnectedComponents(maxIter=5),
}

results = []
for name, fn in tests.items():
    try:
        fn().limit(1).collect()
        results.append((name, "OK", ""))
    except Exception as ex:
        results.append((name, "FAIL", str(ex).split("\n")[0][:120]))

for name, status, err in results:
    print(f"{status:5} {name:20} {err}")